# 30 · Opto — uniform (expert) phase

**Scope.** PPC silencing during expert / pre-shift uniform behaviour. The circuit
prediction is a **null**: once the statistical model is adequate, PPC is
dispensable, so opto should not move behaviour in ChR2+ (het) animals.

**Primary comparison.** HET-Δ vs WT-Δ (Mann–Whitney, 6 vs 4) — isolates the
inhibition from the laser's light/heat artifact; the only adequately powered test
here (floors p≈0.0095 for a clean separation). Within-genotype opto-vs-nonopto
Wilcoxons are **support** (wt n=4 can never reach p<0.05; het n=6 needs all six
consistent, floor p=0.031).

**Unit of inference is the animal.** Stats are pooled *within* animal into one
megasession (`pool_arrays`), computed there, one value per animal, tested
*across* animals. The lag-1 history stats are block-aware — the previous trial
does not bridge session seams. Per-session values (§6) are for plotting only,
never a test.

Every section gives a table and a figure. §4 interrogates the pse result
directly with per-animal opto-vs-nonopto curves.


In [ ]:
from shared_setup import *
from scipy.stats import wilcoxon, mannwhitneyu

from analysis.opto import (
    compute_opto_stats, compute_opto_delta, compute_opto_trajectory,
    compute_opto_comparisons,
)
from plotting.opto import plot_delta_swarm, plot_stat_trajectory
from behav_utils.plotting import plot_comparison           # opto-vs-nonopto curves + Δ + p

# ── config ───────────────────────────────────────────────────────────────────
PHASE      = 'uniform'
STATS      = ['recency', 'pse', 'win_stay', 'lose_shift', 'slope', 'lapse']
PRIMARY    = ['recency', 'pse']
CONDITIONS = {'opto': 'opto', 'nonopto': 'opto_off', 'post': 'post_opto'}
ALPHA      = 0.05
N_BOOT     = 1000

# Genotype is NOT in the data — inject it. Keep in sync as the cohort grows.
# het = ChR2+ (inhibition, effect of interest); wt = transgene-negative control.
GENOTYPE = {
    'SS14': 'wt',  'SS15': 'het', 'SS16': 'het', 'SS17': 'wt',  'SS18': 'wt',
    'SS19': 'het', 'SS20': 'wt',  'SS21': 'het', 'SS22': 'het', 'SS23': 'het',
}

In [ ]:
experiment, info = load_data()
print(f"loaded ({info['mode']}): {experiment.n_animals} animals")

for aid, animal in experiment.animals.items():
    animal.metadata['genotype'] = GENOTYPE.get(aid, 'unknown')

opto_ids = [aid for aid in GENOTYPE if aid in experiment.animals]
het = [a for a in opto_ids if GENOTYPE[a] == 'het']
wt  = [a for a in opto_ids if GENOTYPE[a] == 'wt']
print(f"opto cohort present: {len(opto_ids)}  (het={len(het)} {het}, wt={len(wt)} {wt})")
missing = [aid for aid in GENOTYPE if aid not in experiment.animals]
if missing:
    print(f"WARNING — in GENOTYPE map but not loaded: {missing}")

## §1 · Per-animal aggregation

`compute_opto_stats` pools each animal's uniform opto sessions (`pool_arrays`)
and computes every stat in `STATS`, one value per (animal, condition). Curve
params (`pse`, `slope`, `lapse`) carry a bootstrap CI (the reliability gate);
summary stats carry NaN CIs. `compute_opto_delta` reduces to Δ = opto − nonopto.


In [ ]:
stats_df = compute_opto_stats(
    experiment, phase=PHASE, stats=STATS, conditions=CONDITIONS,
    n_bootstrap=N_BOOT, animals=opto_ids,
)
delta_df = compute_opto_delta(stats_df, opto='opto', nonopto='nonopto')
print(f"stats_df: {stats_df.shape} | delta_df: {delta_df.shape}")
stats_df.head(12)

## §2 · Within-genotype: opto vs non-opto  *(support)*

Paired Wilcoxon of each animal's Δ against zero, per genotype. **Support, not
headline:** wt (n=4) can never reach p<0.05; het (n=6) reaches p=0.031 only if
all six move the same way. The Δ values are visualised in the §3 swarm.


In [ ]:
def within_genotype_wilcoxon(delta_df, stats, genotypes=('het', 'wt')):
    """Paired Wilcoxon of per-animal Δ against zero, per genotype per stat."""
    rows = []
    for g in genotypes:
        for s in stats:
            d = delta_df[(delta_df.genotype == g) & (delta_df.stat == s)].delta.dropna().to_numpy()
            if len(d) >= 1 and np.any(d != 0):
                try:
                    W, p = wilcoxon(d)
                except ValueError:
                    W, p = np.nan, np.nan
            else:
                W, p = np.nan, np.nan
            rows.append(dict(genotype=g, stat=s, n=len(d),
                             median_delta=np.median(d) if len(d) else np.nan, W=W, p=p))
    return pd.DataFrame(rows)

within = within_genotype_wilcoxon(delta_df, STATS)
within

## §3 · Across genotype (primary): HET-Δ vs WT-Δ

Mann–Whitney on the per-animal Δ, het vs wt — the test that isolates inhibition
from the laser artifact. The swarm shows **all** stats; ★ marks the pre-specified
primary readouts. A real effect should appear in the robust stats (recency),
not only in the fit-sensitive ones (pse).


In [ ]:
def genotype_delta_mannwhitney(delta_df, stats, primary=PRIMARY):
    """HET-Δ vs WT-Δ per stat (the primary comparison)."""
    rows = []
    for s in stats:
        hv = delta_df[(delta_df.genotype == 'het') & (delta_df.stat == s)].delta.dropna().to_numpy()
        wv = delta_df[(delta_df.genotype == 'wt')  & (delta_df.stat == s)].delta.dropna().to_numpy()
        U, p = (mannwhitneyu(hv, wv, alternative='two-sided') if len(hv) and len(wv) else (np.nan, np.nan))
        rows.append(dict(stat=s, n_het=len(hv), n_wt=len(wv),
                         med_het=np.median(hv) if len(hv) else np.nan,
                         med_wt=np.median(wv) if len(wv) else np.nan,
                         U=U, p=p, primary=s in primary))
    return pd.DataFrame(rows)

mw = genotype_delta_mannwhitney(delta_df, STATS)
display(mw)

# all-stats Δ swarm: per-animal Δ (opto − nonopto), het vs wt, one panel per stat
ncol = 3; nrow = int(np.ceil(len(STATS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.3 * ncol, 3.5 * nrow))
for ax, s in zip(axes.flat, STATS):
    p = mw.loc[mw.stat == s, 'p'].iloc[0]
    plot_delta_swarm(delta_df, s, ax=ax, p_value=p)
    # if s in PRIMARY:
    #     ax.set_title(f"{s}  ★")
for ax in list(axes.flat)[len(STATS):]:
    ax.set_visible(False)
fig.suptitle('Δ (opto − nonopto) by stat — het vs wt', y=1.01)
fig.tight_layout()

# save the fig as pdf high quality
fig.savefig(f"opto_delta_swarm_{PHASE}.pdf", bbox_inches='tight', dpi=300)

## §4 · Per-animal psychometric diagnostic

The direct check on the pse result. For each animal, `compare_conditions`
overlays the opto and nonopto curves and `plot_comparison` annotates Δpse, Δslope,
Δlapse and **within-animal permutation p-values**. Read two things: does the opto
curve genuinely shift sideways (a true bias) or do the asymptotes differ with a
similar midpoint (mu misreporting), and is any single animal's pse shift
individually significant. If the wt pse shifts are not individually significant,
the across-animal pse result is noise.


In [ ]:
comps = compute_opto_comparisons(
    experiment, phase=PHASE, cond_a='opto', cond_b='nonopto',
    animals=opto_ids, n_permutations=1000, n_bootstrap=1000,
)

ncol = 3; nrow = int(np.ceil(len(comps) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 3.7 * nrow))
for ax, (aid, res) in zip(axes.flat, comps.items()):
    plot_comparison(res, ax=ax,
                    stats_keys=('mu', 'sigma', 'lapse_low', 'lapse_high', 'accuracy'))
    ax.set_title(f"{aid} ({GENOTYPE.get(aid, '?')})")
for ax in list(axes.flat)[len(comps):]:
    ax.set_visible(False)
fig.tight_layout()

In [ ]:
# within-animal pse-shift significance (permutation p on Δmu), sorted by genotype
pse_p = pd.DataFrame([
    dict(animal=aid, genotype=GENOTYPE.get(aid, '?'),
         pse_opto=res['params_a']['mu'], pse_nonopto=res['params_b']['mu'],
         d_pse=res['params_a']['mu'] - res['params_b']['mu'],
         perm_p_pse=res.get('perm_p', {}).get('mu'),
         n_opto=res.get('n_a'), n_nonopto=res.get('n_b'))
    for aid, res in comps.items()
]).sort_values(['genotype', 'animal'])
pse_p

## §5 · Recovery: opto vs post-opto

Does the perturbation persist into the first non-laser trial after each opto run
(opto − post), and is that post trial back to the non-laser baseline
(nonopto − post)? Het only — descriptive, n small.


In [ ]:
# compute_opto_delta differences the two named conditions (minuend, subtrahend)
persist_delta = compute_opto_delta(stats_df, opto='opto',    nonopto='post')  # opto − post
base_delta    = compute_opto_delta(stats_df, opto='nonopto', nonopto='post')  # nonopto − post

persist    = within_genotype_wilcoxon(persist_delta, STATS, genotypes=('het',))
basereturn = within_genotype_wilcoxon(base_delta,    STATS, genotypes=('het',))
display(persist.rename(columns={'median_delta': 'median(opto − post)'}))
display(basereturn.rename(columns={'median_delta': 'median(nonopto − post)'}))

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.7))
for ax, (df, lbl) in zip(axes, [(persist_delta, 'opto − post'), (base_delta, 'nonopto − post')]):
    plot_delta_swarm(df, 'recency', ax=ax, genotype_order=['het', 'wt'])
    ax.set_ylabel(f"Δ recency ({lbl})")
fig.tight_layout()

## §6 · Trajectory over sessions

Per-session stat, one line per animal, het warm / wt cool, opto (solid) vs
nonopto (dashed). Reliable for summary stats; curve-stat trajectories are a
diagnostic only (many per-session fits fail — see the `success` column).


In [ ]:
traj_df = compute_opto_trajectory(
    experiment, phase=PHASE, stats=STATS, conditions=CONDITIONS, animals=opto_ids,
)
fig, ax = plt.subplots(figsize=(6.4, 4.0))
plot_stat_trajectory(traj_df, 'recency', ax=ax, condition='opto', linestyle='-')
plot_stat_trajectory(traj_df, 'recency', ax=ax, condition='nonopto',
                     linestyle='--', show_legend=False)
ax.set_title('recency over sessions — opto (solid) vs nonopto (dashed)')
fig.tight_layout()

## §7 · Goodness-of-fit / QC

Flag curve-stat fits whose bootstrap CI is wide relative to the per-stat median —
unreliable PSE/slope/lapse. The pse CI-width figure is the answer to whether the
§3 pse result rests on solid fits. Threshold (2× median) is a heuristic.


In [ ]:
curve = stats_df[stats_df.stat.isin(['pse', 'slope', 'lapse'])].copy()
curve['ci_width'] = curve['ci_hi'] - curve['ci_lo']
thr = curve.groupby('stat')['ci_width'].transform('median') * 2
curve['flagged'] = curve['ci_width'] > thr
display(curve[curve['flagged']][['animal', 'genotype', 'condition', 'stat',
                                 'value', 'ci_lo', 'ci_hi', 'ci_width', 'n_trials']])

# pse CI width per animal/condition — wide = unreliable fit
pse = curve[curve.stat == 'pse'].copy()
pse['key'] = pse['animal'] + ' / ' + pse['condition']
pse = pse.sort_values('ci_width')
colours = ['#d1495b' if g == 'het' else '#30638e' for g in pse['genotype']]
fig, ax = plt.subplots(figsize=(6, max(3, 0.28 * len(pse))))
ax.barh(pse['key'], pse['ci_width'], color=colours)
ax.axvline(pse['ci_width'].median() * 2, ls='--', c='grey', lw=1, label='2× median')
ax.set_xlabel('pse bootstrap CI width'); ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()